In [4]:
pip install pandas numpy textblob vaderSentiment scikit-learn

  Using cached textblob-0.19.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl.metadata (572 bytes)
Using cached textblob-0.19.0-py3-none-any.whl (624 kB)
Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl (125 kB)

   ---------------------------------------- 2/2 [textblob]

Note: you may need to restart the kernel to use updated packages.


In [8]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as vader
from textblob import TextBlob as tb

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print('ok')

ok


In [9]:
df = pd.read_csv('IMDB_Dataset.csv')
df.head()
print(df)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


In [6]:
# VADER POLARITY ANALYSIS (TEXT TO NUMBER STEP)
def vader_textblob(text):

    sid = vader()
    vad = sid.polarity_scores(text)
    blob = tb(text).sentiment


    return pd.Series({
        'vader_neg': vad['neg'],
        'vader_neu': vad['neu'],
        'vader_pos': vad['pos'],
        'vader_compound': vad['compound'],
        'textblob_polarity': blob.polarity,
        'textblob_subjectivity': blob.subjectivity
    })
X = df['review'].apply(vader_textblob)
y = df['sentiment'].map({'positive': 1, 'negative': 0})


In [9]:
print(X)
# print(y.head())

       vader_neg  vader_neu  vader_pos  vader_compound  textblob_polarity  \
0          0.179      0.756      0.064         -0.9916           0.023433   
1          0.052      0.773      0.176          0.9670           0.109722   
2          0.114      0.688      0.198          0.9519           0.354008   
3          0.125      0.816      0.059         -0.9213          -0.057813   
4          0.050      0.806      0.144          0.9744           0.217952   
...          ...        ...        ...             ...                ...   
49995      0.045      0.765      0.189          0.9886           0.394425   
49996      0.160      0.730      0.110         -0.6837          -0.276190   
49997      0.181      0.704      0.115         -0.9734           0.056984   
49998      0.116      0.804      0.080         -0.8657          -0.048663   
49999      0.118      0.734      0.148          0.6975           0.120000   

       textblob_subjectivity  
0                   0.490369  
1            

In [13]:
# print(X.head())
print(y)

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


In [8]:
#FUNCTIONS DECLARATION

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# X: (n_features, m), Y: (1, m), W: (n_features, 1)
def gradient_descent(W, b, X, Y, epochs, learning_rate):
    m = X.shape[1]
    for i in range(epochs):
        # FORWARD PROPAGATION
        Z = np.dot(W.T, X) + b
        A = sigmoid(Z)

        # LOSS FUNCTION (binary cross-entropy)
        cost = -(1 / m) * np.sum(Y * np.log(A + 1e-8) + (1 - Y) * np.log(1 - A + 1e-8))

        # BACKWARD PROPAGATION
        dZ = A - Y
        dW = (1 / m) * np.dot(X, dZ.T)
        db = (1 / m) * np.sum(dZ)

        # UPDATE
        W -= learning_rate * dW
        b -= learning_rate * db

        if i % 100 == 0 or i == epochs - 1:
            print(f"Epoch {i:04d} | Loss: {cost:.4f}")

    return W, b



In [9]:
# LOGISTIC REGRESSION TRAINING (FROM SCRATCH)

# Train/test split + scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Vectorized shapes: X is (n_features, m), Y is (1, m)
X_train_T = X_train_scaled.T
X_test_T = X_test_scaled.T
y_train_T = y_train.values.reshape(1, -1)
y_test_T = y_test.values.reshape(1, -1)

# Initialize parameters
W = np.zeros((X_train_T.shape[0], 1))
b = 0.0


# Train
W, b = gradient_descent(W, b, X_train_T, y_train_T, epochs=1000, learning_rate=0.1)

# Evaluate
Z_test = np.dot(W.T, X_test_T) + b
A_test = 1 / (1 + np.exp(-Z_test))
y_pred = (A_test >= 0.5).astype(int).T

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))



Epoch 0000 | Loss: 0.6931
Epoch 0100 | Loss: 0.4930
Epoch 0200 | Loss: 0.4856
Epoch 0300 | Loss: 0.4831
Epoch 0400 | Loss: 0.4823
Epoch 0500 | Loss: 0.4819
Epoch 0600 | Loss: 0.4818
Epoch 0700 | Loss: 0.4817
Epoch 0800 | Loss: 0.4817
Epoch 0900 | Loss: 0.4817
Epoch 0999 | Loss: 0.4817
Accuracy: 0.7744
              precision    recall  f1-score   support

           0       0.78      0.77      0.77      5000
           1       0.77      0.78      0.78      5000

    accuracy                           0.77     10000
   macro avg       0.77      0.77      0.77     10000
weighted avg       0.77      0.77      0.77     10000

[[3843 1157]
 [1099 3901]]


Saved model to results/amine_assignment_model.pkl
